# COMP 469 Lab 02 — Solving Problems by Searching

**Length:** 3 hours  
**Topic:** AIMA Chapter 3, Search Algorithms  
**Environment:** This lab is self-contained and uses only the Python standard library.

## What you will build

You will implement and compare search agents for the Romania route-finding problem.

By the end of the lab, you should be able to:

- Formulate a search problem using states, actions, transition model, goal test, and path cost.
- Explain the difference between a state-space graph and a search tree.
- Implement BFS, DFS, uniform-cost search, greedy best-first search, and A*.
- Compare algorithms using completeness, optimality, node expansion, memory/frontier size, and runtime.
- Explain why a heuristic can make search faster and why greedy search can be non-optimal.

## Deliverable

Submit this notebook with:

- All TODOs completed.
- Checkpoints 1–5 answered.
- Experiment output included.
- A final reflection of about one page.

## Environment
python3 -m venv .env                                          
source .env/bin/activate
python3 -m pip install --upgrade pip
python3 -m pip install ipykernel

# Activate the virtual environment and install dependencies
source .venv/bin/activate

## Lab pacing

| Time | Activity |
|---|---|
| 0:00–0:20 | Setup and problem formulation |
| 0:20–0:40 | Core classes: `Problem`, `Node`, `GraphProblem` |
| 0:40–1:10 | Breadth-first search and depth-first search |
| 1:10–1:35 | Uniform-cost search |
| 1:35–2:10 | Greedy best-first search and A* |
| 2:10–2:35 | 8-puzzle heuristic functions |
| 2:35–2:55 | Controlled experiment and comparison |
| 2:55–3:00 | Final reflection and submit |

## Section 1 — Concept review

A search problem has:

- **Initial state:** where the agent starts.
- **Actions:** what the agent can do in a state.
- **Transition model:** what state results from an action.
- **Goal test:** whether the current state satisfies the objective.
- **Path cost:** total cost of the action sequence.

In this lab, each city is treated as an atomic state. The algorithms do not know what a city contains internally; they only know which states connect to which other states and at what cost.

### Checkpoint 1 — Problem formulation — Instructor key

1. Initial state: `Arad`.
2. Goal state: `Bucharest`.
3. Actions from `Arad`: travel to `Zerind`, `Sibiu`, or `Timisoara`.
4. `RESULT(Arad, Sibiu)` returns `Sibiu`.
5. The road cost from `Arad` to `Sibiu` is `140`.
6. It is atomic because each city is treated as an indivisible state; the algorithm does not inspect internal facts about the city.

In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
import heapq
import itertools
import math
import time
from typing import Any, Callable, Dict, Iterable, List, Optional, Tuple


class Problem:
    """Abstract search problem.

    A problem has:
    - initial state
    - goal state or states
    - actions(state)
    - result(state, action)
    - path_cost(cost_so_far, state1, action, state2)
    - optional heuristic h(node)
    """

    def __init__(self, initial: Any, goal: Optional[Any] = None):
        self.initial = initial
        self.goal = goal

    def actions(self, state: Any) -> Iterable[Any]:
        raise NotImplementedError

    def result(self, state: Any, action: Any) -> Any:
        raise NotImplementedError

    def goal_test(self, state: Any) -> bool:
        if isinstance(self.goal, (list, tuple, set)):
            return state in self.goal
        return state == self.goal

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        return cost_so_far + 1

    def h(self, node: "Node") -> float:
        return 0


@dataclass
class Node:
    """A search-tree node.

    state: current state
    parent: previous node
    action: action used to get here
    path_cost: g(n), total cost from initial state
    depth: number of actions from root
    """

    state: Any
    parent: Optional["Node"] = None
    action: Optional[Any] = None
    path_cost: float = 0
    depth: int = 0

    def __post_init__(self):
        if self.parent is not None:
            self.depth = self.parent.depth + 1

    def __lt__(self, other: "Node") -> bool:
        return self.path_cost < other.path_cost

    def child_node(self, problem: Problem, action: Any) -> "Node":
        next_state = problem.result(self.state, action)
        next_cost = problem.path_cost(self.path_cost, self.state, action, next_state)
        return Node(next_state, self, action, next_cost)

    def expand(self, problem: Problem) -> List["Node"]:
        return [self.child_node(problem, action) for action in problem.actions(self.state)]

    def path(self) -> List["Node"]:
        node = self
        result = []
        while node is not None:
            result.append(node)
            node = node.parent
        return list(reversed(result))

    def solution(self) -> List[Any]:
        return [node.action for node in self.path()[1:]]

    def path_states(self) -> List[Any]:
        return [node.state for node in self.path()]


class PriorityQueue:
    """Priority queue used by best-first, UCS, Greedy, and A* search."""

    def __init__(self, f: Callable[[Node], float]):
        self.f = f
        self.heap: List[Tuple[float, int, Node]] = []
        self.counter = itertools.count()

    def append(self, node: Node) -> None:
        heapq.heappush(self.heap, (self.f(node), next(self.counter), node))

    def pop(self) -> Node:
        return heapq.heappop(self.heap)[2]

    def __len__(self) -> int:
        return len(self.heap)


class Graph:
    """Weighted graph using an adjacency dictionary."""

    def __init__(self, graph_dict: Optional[Dict[Any, Dict[Any, float]]] = None, directed: bool = True):
        self.graph_dict: Dict[Any, Dict[Any, float]] = graph_dict or {}
        self.directed = directed
        if not directed:
            self.make_undirected()

    def make_undirected(self) -> None:
        for a in list(self.graph_dict.keys()):
            for b, distance in list(self.graph_dict[a].items()):
                self.connect(b, a, distance)

    def connect(self, a: Any, b: Any, distance: float = 1) -> None:
        self.graph_dict.setdefault(a, {})[b] = distance
        if not self.directed:
            self.graph_dict.setdefault(b, {})[a] = distance

    def get(self, a: Any, b: Optional[Any] = None):
        links = self.graph_dict.setdefault(a, {})
        if b is None:
            return links
        return links.get(b, math.inf)

    def nodes(self) -> List[Any]:
        result = set(self.graph_dict.keys())
        for neighbors in self.graph_dict.values():
            result.update(neighbors.keys())
        return sorted(result)


class GraphProblem(Problem):
    """Route-finding problem on a weighted graph."""

    def __init__(self, initial: Any, goal: Any, graph: Graph, heuristic: Optional[Dict[Any, float]] = None):
        super().__init__(initial, goal)
        self.graph = graph
        self.heuristic = heuristic or {}

    def actions(self, state: Any) -> Iterable[Any]:
        return self.graph.get(state).keys()

    def result(self, state: Any, action: Any) -> Any:
        return action

    def path_cost(self, cost_so_far: float, state1: Any, action: Any, state2: Any) -> float:
        return cost_so_far + self.graph.get(state1, state2)

    def h(self, node: Node) -> float:
        return self.heuristic.get(node.state, 0)


romania_map = Graph(
    {
        "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
        "Bucharest": {"Urziceni": 85, "Pitesti": 101, "Giurgiu": 90, "Fagaras": 211},
        "Craiova": {"Drobeta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
        "Drobeta": {"Mehadia": 75},
        "Eforie": {"Hirsova": 86},
        "Fagaras": {"Sibiu": 99},
        "Hirsova": {"Urziceni": 98},
        "Iasi": {"Vaslui": 92, "Neamt": 87},
        "Lugoj": {"Timisoara": 111, "Mehadia": 70},
        "Oradea": {"Zerind": 71, "Sibiu": 151},
        "Pitesti": {"Rimnicu Vilcea": 97},
        "Rimnicu Vilcea": {"Sibiu": 80},
        "Urziceni": {"Vaslui": 142},
    },
    directed=False,
)

romania_sld_to_bucharest = {
    "Arad": 366,
    "Bucharest": 0,
    "Craiova": 160,
    "Drobeta": 242,
    "Eforie": 161,
    "Fagaras": 176,
    "Giurgiu": 77,
    "Hirsova": 151,
    "Iasi": 226,
    "Lugoj": 244,
    "Mehadia": 241,
    "Neamt": 234,
    "Oradea": 380,
    "Pitesti": 100,
    "Rimnicu Vilcea": 193,
    "Sibiu": 253,
    "Timisoara": 329,
    "Urziceni": 80,
    "Vaslui": 199,
    "Zerind": 374,
}


@dataclass
class SearchReport:
    algorithm: str
    node: Optional[Node]
    expanded: int
    generated: int
    max_frontier: int
    time_ms: float


def breadth_first_graph_search(problem: Problem, return_report: bool = False):
    start = time.perf_counter()
    node = Node(problem.initial)
    expanded = 0
    generated = 1
    max_frontier = 1

    if problem.goal_test(node.state):
        report = SearchReport("BFS", node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
        return report if return_report else node

    frontier = deque([node])
    reached = {problem.initial}

    while frontier:
        node = frontier.popleft()
        expanded += 1

        for child in node.expand(problem):
            generated += 1
            if child.state not in reached:
                if problem.goal_test(child.state):
                    report = SearchReport("BFS", child, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
                    return report if return_report else child
                reached.add(child.state)
                frontier.append(child)

        max_frontier = max(max_frontier, len(frontier))

    report = SearchReport("BFS", None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def depth_first_graph_search(problem: Problem, limit: int = 50, return_report: bool = False):
    start = time.perf_counter()
    frontier = [Node(problem.initial)]
    reached = set()
    expanded = 0
    generated = 1
    max_frontier = 1

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            report = SearchReport("DFS", node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
            return report if return_report else node

        if node.state not in reached and node.depth < limit:
            reached.add(node.state)
            expanded += 1
            children = list(reversed(node.expand(problem)))
            generated += len(children)
            frontier.extend(children)
            max_frontier = max(max_frontier, len(frontier))

    report = SearchReport("DFS", None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def best_first_graph_search(problem: Problem, f: Callable[[Node], float], name: str = "Best-first", return_report: bool = False):
    start = time.perf_counter()
    node = Node(problem.initial)
    frontier = PriorityQueue(f)
    frontier.append(node)

    reached: Dict[Any, Node] = {node.state: node}
    expanded = 0
    generated = 1
    max_frontier = 1

    while frontier:
        node = frontier.pop()

        # Skip stale worse paths that remain in the heap.
        if node.path_cost > reached[node.state].path_cost:
            continue

        if problem.goal_test(node.state):
            report = SearchReport(name, node, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
            return report if return_report else node

        expanded += 1

        for child in node.expand(problem):
            generated += 1
            if child.state not in reached or child.path_cost < reached[child.state].path_cost:
                reached[child.state] = child
                frontier.append(child)

        max_frontier = max(max_frontier, len(frontier))

    report = SearchReport(name, None, expanded, generated, max_frontier, (time.perf_counter() - start) * 1000)
    return report if return_report else None


def uniform_cost_search(problem: Problem, return_report: bool = False):
    return best_first_graph_search(problem, lambda n: n.path_cost, name="UCS", return_report=return_report)


def greedy_best_first_search(problem: Problem, return_report: bool = False):
    return best_first_graph_search(problem, lambda n: problem.h(n), name="Greedy", return_report=return_report)


def astar_search(problem: Problem, return_report: bool = False):
    return best_first_graph_search(problem, lambda n: n.path_cost + problem.h(n), name="A*", return_report=return_report)


def depth_limited_search(problem: Problem, limit: int = 50):
    def recursive_dls(node: Node, limit_remaining: int):
        if problem.goal_test(node.state):
            return node
        if limit_remaining == 0:
            return "cutoff"

        cutoff_occurred = False
        for child in node.expand(problem):
            if child.state in node.path_states():
                continue
            result = recursive_dls(child, limit_remaining - 1)
            if result == "cutoff":
                cutoff_occurred = True
            elif result is not None:
                return result

        return "cutoff" if cutoff_occurred else None

    return recursive_dls(Node(problem.initial), limit)


def iterative_deepening_search(problem: Problem, max_depth: int = 50) -> Optional[Node]:
    for depth in range(max_depth + 1):
        result = depth_limited_search(problem, depth)
        if result != "cutoff":
            return result
    return None


def recursive_best_first_search(problem: Problem, h: Optional[Callable[[Node], float]] = None) -> Optional[Node]:
    heuristic = h or problem.h

    def rbfs(node: Node, f_limit: float) -> Tuple[Optional[Node], float]:
        if problem.goal_test(node.state):
            return node, 0

        successors = []
        current_path_states = set(node.path_states())

        for child in node.expand(problem):
            if child.state not in current_path_states:
                child.f = max(child.path_cost + heuristic(child), getattr(node, "f", 0))
                successors.append(child)

        if not successors:
            return None, math.inf

        while True:
            successors.sort(key=lambda n: n.f)
            best = successors[0]

            if best.f > f_limit:
                return None, best.f

            alternative = successors[1].f if len(successors) > 1 else math.inf
            result, best.f = rbfs(best, min(f_limit, alternative))

            if result is not None:
                return result, best.f

    start = Node(problem.initial)
    start.f = heuristic(start)
    result, _ = rbfs(start, math.inf)
    return result


class SimpleProblemSolvingAgent:
    """Simple problem-solving agent.

    The agent formulates a problem, searches for a sequence of actions, then executes one action per call.
    """

    def __init__(
        self,
        initial_state: Any,
        goal: Any,
        problem_factory: Callable[[Any, Any], Problem],
        search_algorithm: Callable[[Problem], Optional[Node]] = astar_search,
    ):
        self.state = initial_state
        self.goal = goal
        self.problem_factory = problem_factory
        self.search_algorithm = search_algorithm
        self.action_sequence: List[Any] = []

    def update_state(self, percept: Any) -> Any:
        self.state = percept
        return self.state

    def formulate_goal(self, state: Any) -> Any:
        return self.goal

    def formulate_problem(self, state: Any, goal: Any) -> Problem:
        return self.problem_factory(state, goal)

    def __call__(self, percept: Any) -> Optional[Any]:
        state = self.update_state(percept)

        if not self.action_sequence:
            goal = self.formulate_goal(state)
            problem = self.formulate_problem(state, goal)
            solution_node = self.search_algorithm(problem)
            self.action_sequence = solution_node.solution() if solution_node else []

        if self.action_sequence:
            return self.action_sequence.pop(0)

        return None


# 8-puzzle support for heuristic practice.
GOAL_8 = (0, 1, 2, 3, 4, 5, 6, 7, 8)
START_8 = (7, 2, 4, 5, 0, 6, 8, 3, 1)


def misplaced_tiles(state: Tuple[int, ...], goal: Tuple[int, ...] = GOAL_8) -> int:
    return sum(1 for tile, correct in zip(state, goal) if tile != 0 and tile != correct)


def manhattan_distance(state: Tuple[int, ...], goal: Tuple[int, ...] = GOAL_8) -> int:
    total = 0
    for tile in range(1, 9):
        current_index = state.index(tile)
        goal_index = goal.index(tile)
        current_row, current_col = divmod(current_index, 3)
        goal_row, goal_col = divmod(goal_index, 3)
        total += abs(current_row - goal_row) + abs(current_col - goal_col)
    return total


def path_summary(node: Optional[Node]) -> str:
    if node is None:
        return "No solution"
    return f"{' -> '.join(node.path_states())} | cost={node.path_cost} | depth={node.depth}"


def report_table(reports: List[SearchReport]) -> None:
    header = f"{'Algorithm':<12} {'Cost':>6} {'Depth':>6} {'Expanded':>9} {'Generated':>10} {'MaxFrontier':>12} {'Time(ms)':>9}  Path"
    print(header)
    print("-" * len(header))
    for r in reports:
        if r.node is None:
            cost = depth = "-"
            path = "No solution"
        else:
            cost = int(r.node.path_cost)
            depth = r.node.depth
            path = " -> ".join(r.node.path_states())
        print(f"{r.algorithm:<12} {str(cost):>6} {str(depth):>6} {r.expanded:>9} {r.generated:>10} {r.max_frontier:>12} {r.time_ms:>9.3f}  {path}")


def text_bar(label: str, value: float, scale: float = 1.0, width: int = 50) -> str:
    count = int(value / scale)
    count = max(0, min(width, count))
    return f"{label:<12} | {'#' * count} {value}"


def run_romania_demo() -> List[SearchReport]:
    problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)
    reports = [
        breadth_first_graph_search(problem, return_report=True),
        depth_first_graph_search(problem, return_report=True),
        uniform_cost_search(problem, return_report=True),
        greedy_best_first_search(problem, return_report=True),
        astar_search(problem, return_report=True),
    ]
    report_table(reports)
    return reports

## Section 2 — Smoke test the problem classes

Run this after completing TODOs 1–5.

In [ ]:
romania_problem = GraphProblem("Arad", "Bucharest", romania_map, romania_sld_to_bucharest)

print("Initial:", romania_problem.initial)
print("Goal:", romania_problem.goal)
print("Actions from Arad:", list(romania_problem.actions("Arad")))
print("Result of going to Sibiu:", romania_problem.result("Arad", "Sibiu"))
print("Cost Arad -> Sibiu:", romania_problem.path_cost(0, "Arad", "Sibiu", "Sibiu"))
print("Goal test Bucharest:", romania_problem.goal_test("Bucharest"))

## Section 3 — Breadth-first search

BFS expands the shallowest nodes first. It uses a FIFO queue.

Complete TODOs 6–9, then run the cell below.

In [ ]:
bfs_report = breadth_first_graph_search(romania_problem, return_report=True)
report_table([bfs_report])

### Checkpoint 2 — BFS reasoning — Instructor key

1. Expected BFS path: `Arad -> Sibiu -> Fagaras -> Bucharest`.
2. Expected cost: `450`.
3. BFS is not guaranteed to find the cheapest route when road costs differ. It finds a shallowest path by number of actions.
4. Using `pop()` turns the FIFO behavior into LIFO behavior and makes the algorithm behave like DFS.

## Section 4 — Depth-first search

DFS expands the deepest available node first. It uses a LIFO stack.

Complete TODO 10, then run the cell below.

In [ ]:
dfs_report = depth_first_graph_search(romania_problem, return_report=True)
report_table([dfs_report])

### Checkpoint 3 — BFS vs. DFS — Instructor key

1. DFS may find a different path depending on neighbor ordering.
2. DFS often expands a different number of nodes; exact result depends on stack order.
3. DFS is memory-efficient because it mainly stores a path plus remaining siblings, not the whole frontier layer.
4. DFS is not optimal because it returns the first goal it reaches, regardless of path cost.

## Section 5 — Uniform-cost search

Uniform-cost search expands the node with the lowest path cost so far, `g(n)`.

Complete TODO 11, then run the cell below.

In [ ]:
ucs_report = uniform_cost_search(romania_problem, return_report=True)
report_table([ucs_report])

### Checkpoint 4 — Uniform-cost search — Instructor key

1. Expected UCS path: `Arad -> Sibiu -> Rimnicu Vilcea -> Pitesti -> Bucharest`.
2. Expected cost: `418`.
3. UCS waits until pop time because the first generated goal may not be cheapest. A lower-cost path can be discovered later.
4. BFS expands by depth; UCS expands by cumulative path cost.

## Section 6 — Greedy best-first search and A*

Greedy best-first search uses only the heuristic:

`f(n) = h(n)`

A* uses both the path cost so far and the heuristic estimate:

`f(n) = g(n) + h(n)`

Complete TODOs 12–13, then run the cell below.

In [ ]:
greedy_report = greedy_best_first_search(romania_problem, return_report=True)
astar_report = astar_search(romania_problem, return_report=True)

report_table([greedy_report, astar_report])

### Checkpoint 5 — Greedy vs. A* — Instructor key

1. Greedy usually finds `Arad -> Sibiu -> Fagaras -> Bucharest`, cost `450`, not the cheapest route.
2. A* should match UCS on the Bucharest goal with the straight-line-distance heuristic: cost `418`.
3. Greedy can be fast because it follows the state that looks closest to the goal, but it ignores cost already paid.
4. A* adds `g(n)`, the path cost so far, to the heuristic estimate `h(n)`.

## Section 7 — Depth-limited and iterative deepening search

Complete TODO 14, then run the cell below.

In [ ]:
dls_result = depth_limited_search(romania_problem, limit=3)
ids_result = iterative_deepening_search(romania_problem, max_depth=10)

print("DLS limit=3:", path_summary(dls_result if dls_result != "cutoff" else None))
print("IDS:", path_summary(ids_result))

## Section 8 — A simple problem-solving agent

The agent formulates a problem, searches for a solution, then returns one action at a time.

In [ ]:
def romania_problem_factory(state, goal):
    return GraphProblem(state, goal, romania_map, romania_sld_to_bucharest)

agent = SimpleProblemSolvingAgent(
    initial_state="Arad",
    goal="Bucharest",
    problem_factory=romania_problem_factory,
    search_algorithm=astar_search,
)

current_city = "Arad"
for step in range(5):
    action = agent(current_city)
    print(f"Step {step + 1}: at {current_city}, action = {action}")
    if action is None:
        break
    current_city = action

## Section 9 — 8-puzzle heuristic functions

The 8-puzzle is a standard search problem. We will not solve the full puzzle here; instead, you will implement two heuristic functions.

Complete TODOs 15–16.

In [ ]:
print("Start state:", START_8)
print("Goal state: ", GOAL_8)
print("Misplaced tiles:", misplaced_tiles(START_8))
print("Manhattan distance:", manhattan_distance(START_8))

### Checkpoint 6 — Heuristics — Instructor key

1. For the provided state, misplaced tiles is `8`; Manhattan distance is `18`.
2. Manhattan distance is more informative because it estimates how far each tile must move, not merely whether it is wrong.
3. Admissible heuristics avoid overestimation so A* can preserve its cost-optimality guarantee.

## Section 10 — Controlled experiment

Compare BFS, DFS, UCS, Greedy, and A* on several route-finding pairs.

Fill in the short written analysis after running the experiment.

In [ ]:
test_pairs = [
    ("Arad", "Bucharest"),
    ("Sibiu", "Bucharest"),
    ("Timisoara", "Bucharest"),
    ("Neamt", "Craiova"),
    ("Oradea", "Eforie"),
]

all_reports = []

for start_city, goal_city in test_pairs:
    print(f"\n=== {start_city} -> {goal_city} ===")

    # For this lab, the provided heuristic table is only straight-line distance to Bucharest.
    # A* and Greedy are most meaningful when the goal is Bucharest.
    heuristic = romania_sld_to_bucharest if goal_city == "Bucharest" else {}

    problem = GraphProblem(start_city, goal_city, romania_map, heuristic)

    reports = [
        breadth_first_graph_search(problem, return_report=True),
        depth_first_graph_search(problem, return_report=True),
        uniform_cost_search(problem, return_report=True),
    ]

    if goal_city == "Bucharest":
        reports.append(greedy_best_first_search(problem, return_report=True))
        reports.append(astar_search(problem, return_report=True))

    report_table(reports)
    all_reports.extend(reports)

### Text chart: nodes expanded

This avoids external plotting dependencies. Run the cell below after the experiment.

In [ ]:
for report in all_reports:
    print(text_bar(report.algorithm, report.expanded, scale=1))

## Final reflection

Write about one page answering the following:

1. Which algorithm performed best overall for finding low-cost paths?
2. Which algorithm expanded the fewest nodes?
3. Why is "fewest nodes expanded" not always the same as "best path"?
4. What did this lab show about the tradeoff between optimality, time, and memory?
5. How does A* connect the ideas of path cost and heuristic estimates?

Use at least three specific numbers from your own experiment output.

## Optional extension challenge

Choose one if you finish early.

**Option A:** Implement weighted A* with `f(n) = g(n) + w*h(n)` for `w = 1.5` and `w = 2.0`.

**Option B:** Add a new route pair and compare the algorithms.

**Option C:** Modify the Romania graph by closing one road. How does the best route change?